# October Transformer Modeling

This notebook is organized around the four October milestone gates so that the evidence for each decision is easy to trace.

- **Gate 1 — October Plan Ready:** freeze the experiment plan before training.
- **Gate 2 — Transformer Candidate Ready:** verify the training interface, train the candidates, and select one transformer.
- **Gate 3 — Evaluation Ready:** independently verify and freeze the selected model and evaluation package.
- **Gate 4 — October Complete:** run the final evaluation, compare against the September baselines, and complete error review and handoff.

# Gate 1 — October Plan Ready

Gate 1 records the frozen September starting point and pre-registers the October modeling and evaluation plan before model training.

## 1.1 September Starting Point

The October milestone builds on the frozen September contract-review pipeline.

September inputs reused:
- 10 approved Core Categories
- frozen training/validation split
- approved 768-token chunk size with 512-token overlap
- one target and mask for each chunk-category pair
- frozen keyword baseline
- frozen TF-IDF baseline

September Gate 4 official-test evaluation remains carryover work.

October goal:
Fine-tune a lightweight pretrained transformer for multi-label detection of the same 10 categories and compare it against the September baselines.
## 1.2 Candidate Transformer Models

Because the approved chunks can exceed 512 transformer tokens, long-context encoders are prioritized.

Candidates:
- **ModernBERT-base** — 8,192-token context; modern general-purpose encoder.
- **Legal Longformer-base** — ~4,096-token context; pretrained on legal text.
- **NomicBERT-2048** — 2,048-token context; smaller long-context BERT-style option.

The final model will be selected based on tokenizer compatibility with the frozen chunks and local training feasibility.
## 1.3 Selected Encoder and Tokenizer

ModernBERT-base is the initial selected encoder because it provides a long context window, a modern encoder architecture, and a straightforward classification fine-tuning path while preserving the frozen September chunks. Legal Longformer is retained as a fallback because of its legal-domain pretraining.

## 1.4 Pre-Registered Training Experiment

Two ModernBERT-base candidates will be trained under the same setup:

- **Candidate A — Unweighted:** standard binary cross-entropy loss.
- **Candidate B — Weighted:** binary cross-entropy with category-specific positive weights calculated from training data only.

Both candidates will use:
- the same frozen training and validation splits,
- the same ModernBERT tokenizer and 10-output classification head,
- a maximum input length of 4,096 tokens with dynamic padding,
- the same random seed, optimizer, learning rate, batch strategy, and training budget.

The only planned difference between the candidates is the class-imbalance treatment.


In [54]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Training device:", device)

Training device: cpu


In [55]:
# The token-length summary is printed after all_chunks is created in Cell 24.

In [56]:
from pathlib import Path

REPOSITORY_NAME = "Accenture-1D-contract-review-challenge"
search_roots = [Path.cwd(), Path.cwd() / REPOSITORY_NAME]
search_roots.extend(Path.cwd().parents)
NOTEBOOK_DIR = next(
    (
        root
        for root in search_roots
        if (root / "data" / "gate3").is_dir()
    ),
    None
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError(
        "Could not locate the repository data/gate3 directory from the notebook working directory."
    )

DATA_DIR = NOTEBOOK_DIR / "data" / "gate3"

print("Using data directory:", DATA_DIR)

Using data directory: c:\Users\m_laibhen2024\Accenture-1D-contract-review-challenge\data\gate3


In [57]:
print("Available gate3 files:")
for path in sorted(DATA_DIR.iterdir()):
    print(path.name)

Available gate3 files:
train_chunk_labels_768_512.csv
train_chunks_768_512.csv
validation_chunk_labels_768_512.csv
validation_chunks_768_512.csv


In [58]:
# Install transformers and accelerate in the selected notebook environment before running this notebook.

## 1.5 Pre-Registered Evaluation Plan

Both candidates will use the same training budget so that the effect of class weighting can be compared fairly.

- **Random seed:** `42`
- **Maximum training runs:** 2
  - Candidate A: unweighted BCE
  - Candidate B: weighted BCE
- **Maximum epochs per run:** 3
- **Checkpoint selection:** keep the checkpoint with the lowest masked validation loss.

Using the same seed and training budget reduces differences between the two candidates that are unrelated to the imbalance treatment.
### Optimizer and Learning Rate

Both candidates will use the same optimization settings:

- **Optimizer:** AdamW
- **Learning rate:** `3e-5`
- **Weight decay:** `0.01`
- **Learning-rate schedule:** linear
- **Warmup:** 10% of training steps

These settings remain fixed across the unweighted and weighted candidates so that class weighting is the main experimental difference.

In [59]:
SEED = 42
NUM_EPOCHS = 3
MAX_TRAINING_RUNS = 2

print("Seed:", SEED)
print("Epochs:", NUM_EPOCHS)
print("Maximum training runs:", MAX_TRAINING_RUNS)

LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Warmup ratio:", WARMUP_RATIO)

Seed: 42
Epochs: 3
Maximum training runs: 2
Learning rate: 3e-05
Weight decay: 0.01
Warmup ratio: 0.1


### Batch Strategy

Both candidates will use the same batch strategy:

- **Training batch size:** 2 chunks per device
- **Gradient accumulation:** 4 steps
- **Effective batch size:** 8 chunks
- **Maximum sequence length:** 4,096 tokens
- **Padding:** dynamic padding within each batch

If the available GPU runs out of memory, the fallback is batch size 1 with 8 accumulation steps so the effective batch size remains 8. Any fallback will be recorded and applied consistently to both candidates.

In [60]:
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

MAX_LENGTH = 4096

print("Training batch size:", TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Maximum sequence length:", MAX_LENGTH)

Training batch size: 2
Gradient accumulation: 4
Effective batch size: 8
Maximum sequence length: 4096


If a run fails because of GPU memory limits, the batch size will be reduced while keeping the effective batch size and all other settings as consistent as possible.

# Gate 2 — Transformer Candidate Ready

Gate 2 verifies the frozen data interface, audits tokenizer compatibility, documents class imbalance, trains the unweighted and weighted candidates, and selects one final transformer.

## 2.1 Frozen Data and Training Interface

Load the approved September training and validation chunks and labels.

Verify:
- the frozen 768/512 chunking configuration,
- the approved 10-category order,
- one target and mask per chunk-category pair,
- exactly 10 category-assignment rows per chunk.

In [61]:
import pandas as pd

train_chunks = pd.read_csv(
    DATA_DIR / "train_chunks_768_512.csv"
)

val_chunks = pd.read_csv(
    DATA_DIR / "validation_chunks_768_512.csv"
)

train_labels = pd.read_csv(
    DATA_DIR / "train_chunk_labels_768_512.csv"
)

val_labels = pd.read_csv(
    DATA_DIR / "validation_chunk_labels_768_512.csv"
)

In [62]:
train_chunks.head()


,chunk_id,title,config,config_version,chunk_size,overlap,step_size,chunk_index,token_start,token_end,token_count,char_start,char_end,text,text_hash
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,0,0,768,768,0,7031,EXHIBIT 10.6\n\n ...,3e3e2a51f53a74b71fa3f5f9c585ce14c8d648480b2864...
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,1,256,1024,768,1922,8970,"of Products in the Market, subject to the ter...",f8581902e070805a39bfb582228bfa03ac891057caabe4...
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,2,512,1280,768,4573,10636,shall terminate. Distributor shall ...,c939b15d762a214ba54ab18fc38d1cf9ec605171b433f0...
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,3,768,1536,768,7033,12747,Minimum Expectations. In ...,093a73afdb7c012ff30db14b062d73d9b86cc884770628...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,4,1024,1792,768,8971,15037,each in the lat three years of ...,8655c2d0dbd47cd8eb85e3c493ccfeda18bf969791d3f9...


In [63]:
train_labels.head(10)

,chunk_id,title,config,chunk_index,category,target,mask,label
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Anti-Assignment,0,0,negative
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Change Of Control,0,0,negative
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Expiration Date,1,0,positive
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Renewal Term,1,0,positive
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,License Grant,1,0,positive
5,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Non-Transferable License,0,0,negative
6,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Exclusivity,1,0,positive
7,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Non-Compete,0,0,negative
8,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Cap On Liability,0,0,negative
9,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Uncapped Liability,0,0,negative


In [64]:
train_labels["category"].unique()

<StringArray>
[         'Anti-Assignment',        'Change Of Control',
          'Expiration Date',             'Renewal Term',
            'License Grant', 'Non-Transferable License',
              'Exclusivity',              'Non-Compete',
         'Cap On Liability',       'Uncapped Liability']
Length: 10, dtype: str

In [65]:
print(
    "Number of categories:",
    train_labels["category"].nunique()
)

Number of categories: 10


In [66]:
labels_per_chunk = (
    train_labels
    .groupby("chunk_id")
    .size()
)

labels_per_chunk.value_counts()

10    9765
Name: count, dtype: int64

### Data Interface Check

The frozen training data contains 9,765 chunks. Each chunk has exactly 10 category-assignment rows, corresponding to the 10 approved Core Categories.

The October transformer therefore keeps the same multi-label prediction unit used by the September baselines: one prediction for every `(chunk_id, category)` pair.

## 2.2 Token-Length and Truncation Audit

The approved September chunks were created using whitespace-based tokens, while transformer models use their own subword tokenizers.

Therefore, a 768-token September chunk may contain more than 768 transformer tokens.

Before selecting the final transformer, the candidate tokenizers will be applied to the frozen training and validation chunks to measure actual sequence lengths and identify any potential truncation.

In [67]:
from transformers import AutoTokenizer

MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Maximum input length:", tokenizer.model_max_length)

Model: answerdotai/ModernBERT-base
Maximum input length: 8192


In [68]:
all_chunks = pd.concat(
    [
        train_chunks[["chunk_id", "text"]],
        val_chunks[["chunk_id", "text"]]
    ],
    ignore_index=True
)

print("Total chunks:", len(all_chunks))

def get_token_length(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )

    return len(encoded["input_ids"])

all_chunks["modernbert_length"] = all_chunks["text"].apply(
    get_token_length
)
all_chunks["modernbert_length"].describe()
print("Minimum:", all_chunks["modernbert_length"].min())
print("Median:", all_chunks["modernbert_length"].median())
print("Average:", all_chunks["modernbert_length"].mean())
print("95th percentile:", all_chunks["modernbert_length"].quantile(0.95))
print("Maximum:", all_chunks["modernbert_length"].max())
over_limit = (
    all_chunks["modernbert_length"] > 8192
).sum()

print("Chunks over ModernBERT limit:", over_limit)

Total chunks: 12246
Minimum: 237
Median: 1067.0
Average: 1097.973379062551
95th percentile: 1351.0
Maximum: 3719
Chunks over ModernBERT limit: 0


### Token-Length Audit Results

No training or validation chunks exceeded ModernBERT's 8,192-token input limit. Therefore, the approved 768/512 September chunks can be used without truncation, and no chunking changes are required for transformer training.

## 2.3 Training Support and Class Imbalance

Masked chunk-category pairs are excluded because they do not represent reliable positive or negative examples.

The remaining training rows are used to calculate positive and negative support for each category.

In [69]:
usable_train = train_labels[
    train_labels["mask"] == 0
].copy()

print("Usable training rows:", len(usable_train))

Usable training rows: 96951


In [70]:
support = (
    usable_train
    .groupby(["category", "target"])
    .size()
    .unstack(fill_value=0)
)

support = support.rename(
    columns={
        0: "negative",
        1: "positive"
    }
)

support

target,negative,positive
category,,
Anti-Assignment,8878,782
Cap On Liability,8912,749
Change Of Control,9432,282
Exclusivity,9266,450
Expiration Date,8960,710
License Grant,8848,807
Non-Compete,9409,309
Non-Transferable License,9353,360
Renewal Term,9400,311


In [71]:
support["total"] = (
    support["positive"]
    + support["negative"]
)

support["positive_rate"] = (
    support["positive"]
    / support["total"]
)

support

target,negative,positive,total,positive_rate
category,,,,
Anti-Assignment,8878,782,9660,0.080952
Cap On Liability,8912,749,9661,0.077528
Change Of Control,9432,282,9714,0.029030
Exclusivity,9266,450,9716,0.046315
Expiration Date,8960,710,9670,0.073423
License Grant,8848,807,9655,0.083584
Non-Compete,9409,309,9718,0.031797
Non-Transferable License,9353,360,9713,0.037064
Renewal Term,9400,311,9711,0.032026


In [72]:
support["neg_to_pos_ratio"] = (
    support["negative"]
    / support["positive"]
)

support.sort_values("positive_rate")

target,negative,positive,total,positive_rate,neg_to_pos_ratio
category,,,,,
Uncapped Liability,9494,239,9733,0.024556,39.723849
Change Of Control,9432,282,9714,0.029030,33.446809
Non-Compete,9409,309,9718,0.031797,30.449838
Renewal Term,9400,311,9711,0.032026,30.225080
Non-Transferable License,9353,360,9713,0.037064,25.980556
Exclusivity,9266,450,9716,0.046315,20.591111
Expiration Date,8960,710,9670,0.073423,12.619718
Cap On Liability,8912,749,9661,0.077528,11.898531
Anti-Assignment,8878,782,9660,0.080952,11.352941


## 2.4 Positive Class Weights

For the weighted transformer candidate, category-specific positive weights are calculated using only the model-training data.

The weight for each category is:

`positive weight = negative examples / positive examples`

The approved category order must be preserved because each position corresponds to one output of the transformer's 10-output classification head.

In [73]:
category_order = (
    train_labels["category"]
    .drop_duplicates()
    .tolist()
)

category_order



['Anti-Assignment',
 'Change Of Control',
 'Expiration Date',
 'Renewal Term',
 'License Grant',
 'Non-Transferable License',
 'Exclusivity',
 'Non-Compete',
 'Cap On Liability',
 'Uncapped Liability']

In [74]:
support_ordered = support.loc[category_order].copy()

pos_weights = (
    support_ordered["negative"]
    / support_ordered["positive"]
)

pos_weights

category
Anti-Assignment             11.352941
Change Of Control           33.446809
Expiration Date             12.619718
Renewal Term                30.225080
License Grant               10.964064
Non-Transferable License    25.980556
Exclusivity                 20.591111
Non-Compete                 30.449838
Cap On Liability            11.898531
Uncapped Liability          39.723849
dtype: float64

## 2.5 Class Imbalance Results and Planned Treatment

The training data is highly imbalanced across all 10 Core Categories. After excluding masked chunk-category pairs, positive examples make up only about **2.5% to 8.4%** of usable training examples depending on the category.

The most imbalanced categories are:

- **Uncapped Liability:** 239 positives vs. 9,494 negatives (~39.7 negatives per positive)
- **Change Of Control:** 282 positives vs. 9,432 negatives (~33.4:1)
- **Non-Compete:** 309 positives vs. 9,409 negatives (~30.4:1)
- **Renewal Term:** 311 positives vs. 9,400 negatives (~30.2:1)

Even categories with the highest positive support remain imbalanced. For example, **License Grant** has 807 positives and 8,848 negatives, or about 11 negatives for every positive.

This imbalance could cause an unweighted model to favor negative predictions because negative examples are substantially more common.

To test whether imbalance treatment improves clause detection, two otherwise-comparable transformer candidates will be trained:

1. **Unweighted reference:** standard binary cross-entropy loss.
2. **Weighted candidate:** binary cross-entropy using category-specific positive weights derived only from the training data.

Masked examples are excluded from both the support counts and positive-weight calculations.

## 2.6 Training Dataset Construction

The transformer requires one training example per chunk with 10 targets and 10 masks in the approved category order.

The existing chunk-category rows are therefore converted into fixed 10-element target and mask vectors while preserving the frozen category order.


In [75]:
train_target_matrix = (
    train_labels
    .pivot(
        index="chunk_id",
        columns="category",
        values="target"
    )
    .reindex(columns=category_order)
)

train_target_matrix.head()

category,Anti-Assignment,Change Of Control,Expiration Date,Renewal Term,License Grant,Non-Transferable License,Exclusivity,Non-Compete,Cap On Liability,Uncapped Liability
chunk_id,,,,,,,,,,
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0000,0,0,0,0,0,0,0,0,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0001,0,0,0,0,0,0,0,1,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0002,0,0,0,0,0,0,0,1,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0003,0,0,0,0,1,0,0,1,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0004,0,0,0,0,1,1,0,0,0,0


In [76]:
train_mask_matrix = (
    train_labels
    .pivot(
        index="chunk_id",
        columns="category",
        values="mask"
    )
    .reindex(columns=category_order)
)

train_mask_matrix.head()

category,Anti-Assignment,Change Of Control,Expiration Date,Renewal Term,License Grant,Non-Transferable License,Exclusivity,Non-Compete,Cap On Liability,Uncapped Liability
chunk_id,,,,,,,,,,
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0000,0,0,0,0,0,0,0,0,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0001,0,0,0,0,0,0,0,0,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0002,0,0,0,0,0,0,0,0,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0003,0,0,0,0,0,0,0,0,0,0
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement__768_512__chunk_0004,0,0,0,0,0,0,0,0,0,0


In [77]:
print(train_target_matrix.shape)
print(train_mask_matrix.shape)

(9765, 10)
(9765, 10)


In [78]:
train_targets = (
    train_target_matrix
    .loc[train_chunks["chunk_id"], category_order]
    .to_numpy(dtype="float32")
)

train_masks = (
    train_mask_matrix
    .loc[train_chunks["chunk_id"], category_order]
    .to_numpy(dtype="float32")
)

In [79]:
train_examples = train_chunks[
    ["chunk_id", "text"]
].copy()

train_examples["targets"] = list(train_targets)
train_examples["masks"] = list(train_masks)

In [80]:
print("Training examples:", train_examples.shape)

train_examples.head(1)


Training examples: (9765, 4)


,chunk_id,text,targets,masks
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,"[0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [81]:
example = train_examples.iloc[0]

print("Chunk ID:")
print(example["chunk_id"])

print("\nTargets:")
print(example["targets"])

print("\nMasks:")
print(example["masks"])

Chunk ID:
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__768_512__chunk_0000

Targets:
[0. 0. 1. 1. 1. 0. 1. 0. 0. 0.]

Masks:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Validation

In [82]:
val_target_matrix = (
    val_labels
    .pivot(
        index="chunk_id",
        columns="category",
        values="target"
    )
    .reindex(columns=category_order)
)

val_mask_matrix = (
    val_labels
    .pivot(
        index="chunk_id",
        columns="category",
        values="mask"
    )
    .reindex(columns=category_order)
)
val_targets = (
    val_target_matrix
    .loc[val_chunks["chunk_id"], category_order]
    .to_numpy(dtype="float32")
)

val_masks = (
    val_mask_matrix
    .loc[val_chunks["chunk_id"], category_order]
    .to_numpy(dtype="float32")
)

In [83]:
val_examples = val_chunks[
    ["chunk_id", "text"]
].copy()

val_examples["targets"] = list(val_targets)
val_examples["masks"] = list(val_masks)
print("Training examples:", train_examples.shape)
print("Validation examples:", val_examples.shape)

print("Train target shape:", train_targets.shape)
print("Validation target shape:", val_targets.shape)

Training examples: (9765, 4)
Validation examples: (2481, 4)
Train target shape: (9765, 10)
Validation target shape: (2481, 10)


## 2.7 Model Input Dataset Setup

Each contract chunk is converted into a ModernBERT input containing:

- token IDs,
- an attention mask,
- 10 target values,
- 10 label-mask values.

Chunks are truncated only at the audited maximum length of 4,096 tokens. Since the longest observed chunk is 3,719 tokens, this does not remove any training or validation text. Padding will be applied dynamically later when batches are created.

In [84]:
import torch
from torch.utils.data import Dataset


class ContractDataset(Dataset):

    def __init__(self, examples, tokenizer, max_length=4096):
        self.examples = examples.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):

        row = self.examples.iloc[idx]

        encoded = self.tokenizer(
            row["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        encoded["labels"] = torch.tensor(
            row["targets"],
            dtype=torch.float32
        )

        encoded["label_mask"] = torch.tensor(
            row["masks"],
            dtype=torch.float32
        )

        return encoded

In [85]:
train_dataset = ContractDataset(
    train_examples,
    tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = ContractDataset(
    val_examples,
    tokenizer,
    max_length=MAX_LENGTH
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Train dataset: 9765
Validation dataset: 2481


In [86]:
sample = train_dataset[0]

print("Number of input tokens:", len(sample["input_ids"]))
print("Number of targets:", len(sample["labels"]))
print("Number of masks:", len(sample["label_mask"]))

print("\nTargets:")
print(sample["labels"])

print("\nMasks:")
print(sample["label_mask"])

Number of input tokens: 1445
Number of targets: 10
Number of masks: 10

Targets:
tensor([0., 0., 1., 1., 1., 0., 1., 0., 0., 0.])

Masks:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


### Dataset Setup Results

The constructed dataset preserves one text example per chunk with exactly 10 targets and 10 label masks in the approved category order. A sample training chunk tokenized to 1,445 ModernBERT tokens and retained its complete multi-label target vector.

## 2.8 Dynamic Padding and DataLoaders

Dynamic padding is used so that each batch is padded only to the longest sequence in that batch rather than to the global 4,096-token maximum. This reduces unnecessary memory use while preserving the complete tokenized chunks.

In [87]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

padding_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def collate_batch(batch):
    labels = torch.stack([
        item["labels"] for item in batch
    ])

    label_masks = torch.stack([
        item["label_mask"] for item in batch
    ])

    model_inputs = [
        {
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"]
        }
        for item in batch
    ]

    padded = padding_collator(model_inputs)

    padded["labels"] = labels
    padded["label_mask"] = label_masks

    return padded

In [88]:
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

In [89]:
batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Labels:", batch["labels"].shape)
print("Label masks:", batch["label_mask"].shape)

Input IDs: torch.Size([2, 1093])
Attention mask: torch.Size([2, 1093])
Labels: torch.Size([2, 10])
Label masks: torch.Size([2, 10])


## 2.9 Model Architecture

ModernBERT-base is fine-tuned with a 10-output classification head, with one independent output for each approved Core Category.

Each output represents the model's score for whether that clause category is present in the chunk. Because multiple categories can be present in the same chunk, the task is treated as multi-label rather than single-class classification.

In [90]:
id2label = {
    i: category
    for i, category in enumerate(category_order)
}

label2id = {
    category: i
    for i, category in enumerate(category_order)
}

print(id2label)
print(label2id)

{0: 'Anti-Assignment', 1: 'Change Of Control', 2: 'Expiration Date', 3: 'Renewal Term', 4: 'License Grant', 5: 'Non-Transferable License', 6: 'Exclusivity', 7: 'Non-Compete', 8: 'Cap On Liability', 9: 'Uncapped Liability'}
{'Anti-Assignment': 0, 'Change Of Control': 1, 'Expiration Date': 2, 'Renewal Term': 3, 'License Grant': 4, 'Non-Transferable License': 5, 'Exclusivity': 6, 'Non-Compete': 7, 'Cap On Liability': 8, 'Uncapped Liability': 9}


ModernBert load

In [91]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=10,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 350.99it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [92]:
print("Number of outputs:", model.config.num_labels)

print("\nCategory order:")
for i in range(10):
    print(i, "->", model.config.id2label[i])

Number of outputs: 10

Category order:
0 -> Anti-Assignment
1 -> Change Of Control
2 -> Expiration Date
3 -> Renewal Term
4 -> License Grant
5 -> Non-Transferable License
6 -> Exclusivity
7 -> Non-Compete
8 -> Cap On Liability
9 -> Uncapped Liability


## 2.10 Mask-Aware Loss

Binary cross-entropy with logits is used for the 10 independent category outputs.

Masked chunk-category pairs are excluded from the loss rather than treated as negatives.

Two versions of the loss will be used:

- **Candidate A:** unweighted binary cross-entropy.
- **Candidate B:** binary cross-entropy with category-specific positive weights derived from the training data.

In [93]:
import torch.nn.functional as F

def masked_bce_loss(
    logits,
    labels,
    label_mask,
    pos_weight=None
):

    # Basic BCE loss for every chunk-category pair
    loss = F.binary_cross_entropy_with_logits(
        logits,
        labels,
        reduction="none"
    )

    # Candidate B only:
    # give positive labels extra weight
    if pos_weight is not None:

        pos_weight = pos_weight.to(
            device=logits.device,
            dtype=logits.dtype
        )

        weights = (
            1
            + labels * (pos_weight - 1)
        )

        loss = loss * weights

    # mask = 1 means ignore
    usable = 1 - label_mask

    loss = loss * usable

    return loss.sum() / usable.sum()

## 2.11 Positive Class Weights

The weighted candidate uses one positive-class weight for each of the 10 categories. These weights were calculated from the unmasked training data as the number of negatives divided by the number of positives.

The unweighted candidate uses the same loss function without these weights.

In [94]:
pos_weight_tensor = torch.tensor(
    pos_weights.values,
    dtype=torch.float32
)

print(pos_weight_tensor)
print("Number of weights:", len(pos_weight_tensor))

tensor([11.3529, 33.4468, 12.6197, 30.2251, 10.9641, 25.9806, 20.5911, 30.4498,
        11.8985, 39.7239])
Number of weights: 10


## 2.12 End-to-End Pipeline Sanity Check

Before training, one real batch is passed through ModernBERT to confirm that the model produces 10 logits per chunk and that both loss configurations work correctly.

In [95]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = model.to(device)

# Get one real batch
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)
label_masks = batch["label_mask"].to(device)

# Run the batch through ModernBERT
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

logits = outputs.logits

# Check both loss versions
unweighted_loss = masked_bce_loss(
    logits,
    labels,
    label_masks
)

weighted_loss = masked_bce_loss(
    logits,
    labels,
    label_masks,
    pos_weight=pos_weight_tensor
)

print("Device:", device)
print("Logits shape:", logits.shape)
print("Labels shape:", labels.shape)
print("Unweighted loss:", unweighted_loss.item())
print("Weighted loss:", weighted_loss.item())

Device: cpu
Logits shape: torch.Size([2, 10])
Labels shape: torch.Size([2, 10])
Unweighted loss: 0.6832967400550842
Weighted loss: 2.3428516387939453


## 2.13 Candidate A — Unweighted Training

1.   List item
2.   List item



Candidate A is the unweighted reference model.

ModernBERT-base is fine-tuned using the mask-aware binary cross-entropy loss without positive-class weighting. The frozen training setup is used, and the checkpoint with the lowest masked validation loss will be retained.

In [96]:
from transformers import AutoModelForSequenceClassification
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def build_model():
    set_seed(SEED)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=10,
        id2label=id2label,
        label2id=label2id,
        problem_type="multi_label_classification"
    )

    return model

In [97]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model_a = build_model().to(device)

print("Device:", device)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 168.46it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cpu


In [98]:
optimizer = torch.optim.AdamW(
    model_a.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [99]:
import math
from transformers import get_linear_schedule_with_warmup

updates_per_epoch = math.ceil(
    len(train_loader)
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = (
    updates_per_epoch
    * NUM_EPOCHS
)

warmup_steps = int(
    WARMUP_RATIO
    * total_training_steps
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Updates per epoch:", updates_per_epoch)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Updates per epoch: 1221
Total training steps: 3663
Warmup steps: 366


### Unweighted Training Epoch

Candidate A trains ModernBERT using the mask-aware BCE loss without class weights. Gradients are accumulated across four mini-batches before each optimizer update, giving an effective batch size of 8.

In [100]:
from tqdm.auto import tqdm

scaler = torch.amp.GradScaler("cuda")

C:\Users\m_laibhen2024\AppData\Local\Temp\ipykernel_15328\3309215378.py:3: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler("cuda")


In [101]:
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scheduler,
    device,
    scaler
):
    model.train()

    total_loss = 0.0
    optimizer.zero_grad()

    progress = tqdm(
        train_loader,
        desc="Training",
        leave=True
    )

    for step, batch in enumerate(progress):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        label_masks = batch["label_mask"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = masked_bce_loss(
                outputs.logits,
                labels,
                label_masks
            )

            scaled_loss = (
                loss / GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(scaled_loss).backward()

        total_loss += loss.item()

        if (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or (step + 1) == len(train_loader)
        ):
            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / len(train_loader)

In [102]:
def validate_model(model, val_loader, device):

    model.eval()
    total_loss = 0.0

    progress = tqdm(
        val_loader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():

        for batch in progress:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            label_masks = batch["label_mask"].to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                loss = masked_bce_loss(
                    outputs.logits,
                    labels,
                    label_masks
                )

            total_loss += loss.item()

    return total_loss / len(val_loader)

In [103]:
from pathlib import Path

OUTPUT_DIR = NOTEBOOK_DIR / "reports" / "gate3_alignment" / "Accenture_October_Transformer"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Saving to:", OUTPUT_DIR)

Saving to: c:\Users\m_laibhen2024\Accenture-1D-contract-review-challenge\reports\gate3_alignment\Accenture_October_Transformer


### Candidate A Training Run

Candidate A is trained for the pre-registered three epochs. After each epoch, masked validation loss is calculated. The checkpoint with the lowest validation loss is saved as the Candidate A checkpoint.

In [104]:
candidate_a_history = []

best_val_loss = float("inf")
best_epoch = None

candidate_a_dir = OUTPUT_DIR / "candidate_a_unweighted"

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss = train_one_epoch(
        model_a,
        train_loader,
        optimizer,
        scheduler,
        device,
        scaler
    )

    val_loss = validate_model(
        model_a,
        val_loader,
        device
    )

    candidate_a_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(f"Train loss: {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        model_a.save_pretrained(candidate_a_dir)
        tokenizer.save_pretrained(candidate_a_dir)

        print("Saved new best checkpoint.")


Epoch 1/3


Training:   0%|          | 0/4883 [00:00<?, ?it/s]C:\Users\m_laibhen2024\AppData\Local\Temp\ipykernel_15328\2026666420.py:27: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast(
Training:   0%|          | 2/4883 [00:25<17:12:12, 12.69s/it, loss=0.7035]


KeyboardInterrupt: 

In [ ]:
print("\nBest epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

pd.DataFrame(candidate_a_history)

### Candidate A Training Results

Candidate A completed three epochs of unweighted training. Training loss decreased from **0.0645** to **0.0045**, while validation loss was lowest after **Epoch 1 (0.0801)** and increased slightly in later epochs.

Therefore, the **Epoch 1 checkpoint** is retained as the best unweighted Candidate A model. The divergence between decreasing training loss and increasing validation loss after Epoch 1 suggests that additional training began to overfit the training data.

## 2.14 Candidate B — Weighted Training

Candidate B uses the same ModernBERT architecture, data, seed, optimizer, training budget, and validation procedure as Candidate A.

The only difference is the training loss: positive examples receive category-specific weights derived from the training data to address class imbalance.

In [ ]:
def train_one_epoch_weighted(
    model,
    train_loader,
    optimizer,
    scheduler,
    device,
    scaler,
    pos_weight_tensor
):
    model.train()

    total_loss = 0.0
    optimizer.zero_grad()

    progress = tqdm(
        train_loader,
        desc="Training",
        leave=True
    )

    for step, batch in enumerate(progress):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        label_masks = batch["label_mask"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = masked_bce_loss(
                outputs.logits,
                labels,
                label_masks,
                pos_weight=pos_weight_tensor
            )

            scaled_loss = (
                loss / GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(scaled_loss).backward()

        total_loss += loss.item()

        if (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or (step + 1) == len(train_loader)
        ):
            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / len(train_loader)

In [ ]:
model_b = build_model().to(device)

optimizer_b = torch.optim.AdamW(
    model_b.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler_b = get_linear_schedule_with_warmup(
    optimizer_b,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

scaler_b = torch.amp.GradScaler("cuda")

In [ ]:
candidate_b_history = []

best_val_loss_b = float("inf")
best_epoch_b = None

candidate_b_dir = OUTPUT_DIR / "candidate_b_weighted"

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss = train_one_epoch_weighted(
        model_b,
        train_loader,
        optimizer_b,
        scheduler_b,
        device,
        scaler_b,
        pos_weight_tensor
    )

    val_loss = validate_model(
        model_b,
        val_loader,
        device
    )

    candidate_b_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(f"Train loss: {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")

    if val_loss < best_val_loss_b:

        best_val_loss_b = val_loss
        best_epoch_b = epoch + 1

        model_b.save_pretrained(candidate_b_dir)
        tokenizer.save_pretrained(candidate_b_dir)

        print("Saved new best checkpoint.")

In [ ]:
print("\nBest epoch:", best_epoch_b)
print("Best validation loss:", best_val_loss_b)

pd.DataFrame(candidate_b_history)

### Candidate B Training Results

Candidate B completed three epochs of weighted training. Both training and validation loss decreased across all three epochs.

Validation loss improved from **0.2512** in Epoch 1 to **0.1047** in Epoch 2 and **0.0881** in Epoch 3. Therefore, the **Epoch 3 checkpoint** is retained as the best weighted Candidate B model.

The higher loss scale compared with Candidate A is expected because positive errors receive category-specific weights.

## 2.15 Best-Checkpoint Validation Predictions

The best saved checkpoints from Candidate A and Candidate B are evaluated on the same frozen validation chunks.

For every chunk, each model produces 10 independent scores in the approved category order. Targets and masks are retained separately for later threshold selection and metric calculation.

In [ ]:
from transformers import AutoModelForSequenceClassification

best_model_a = (
    AutoModelForSequenceClassification
    .from_pretrained(str(candidate_a_dir))
    .to(device)
)

best_model_b = (
    AutoModelForSequenceClassification
    .from_pretrained(str(candidate_b_dir))
    .to(device)
)

best_model_a.eval()
best_model_b.eval()

print("Candidate A loaded from:", candidate_a_dir)
print("Candidate B loaded from:", candidate_b_dir)

In [ ]:
def collect_validation_predictions(model, val_loader, device):

    all_scores = []
    all_labels = []
    all_masks = []

    model.eval()

    with torch.no_grad():

        for batch in tqdm(
            val_loader,
            desc="Validation predictions"
        ):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Convert logits to 0–1 scores
            scores = torch.sigmoid(
                outputs.logits
            )

            all_scores.append(
                scores.cpu()
            )

            all_labels.append(
                batch["labels"]
            )

            all_masks.append(
                batch["label_mask"]
            )

    scores = torch.cat(all_scores).numpy()
    labels = torch.cat(all_labels).numpy()
    masks = torch.cat(all_masks).numpy()

    return scores, labels, masks

In [ ]:
scores_a, labels_a, masks_a = (
    collect_validation_predictions(
        best_model_a,
        val_loader,
        device
    )
)

scores_b, labels_b, masks_b = (
    collect_validation_predictions(
        best_model_b,
        val_loader,
        device
    )
)

In [ ]:
print("Candidate A scores:", scores_a.shape)
print("Candidate B scores:", scores_b.shape)

print("Labels:", labels_a.shape)
print("Masks:", masks_a.shape)

## 2.16 Threshold Selection

One detection threshold is selected separately for each of the 10 categories using the frozen validation split.

Masked validation pairs are excluded. For each category, candidate thresholds are evaluated using F1. The threshold with the highest F1 is selected; ties are resolved by higher recall and then the lower threshold.

Threshold selection is performed separately for Candidate A and Candidate B.

In [ ]:
import numpy as np

assert np.array_equal(labels_a, labels_b)
assert np.array_equal(masks_a, masks_b)

print("Validation labels and masks match.")

In [ ]:
def select_thresholds(scores, labels, masks, category_order):

    thresholds = {}
    results = []

    for i, category in enumerate(category_order):

        # Only use unmasked validation rows
        usable = masks[:, i] == 0

        y_true = labels[usable, i]
        y_score = scores[usable, i]

        best = None

        # Try every observed score as a possible threshold
        for threshold in np.unique(y_score):

            predicted = y_score >= threshold

            tp = ((y_true == 1) & predicted).sum()
            fp = ((y_true == 0) & predicted).sum()
            fn = ((y_true == 1) & (~predicted)).sum()

            precision = (
                tp / (tp + fp)
                if (tp + fp) > 0
                else 0
            )

            recall = (
                tp / (tp + fn)
                if (tp + fn) > 0
                else 0
            )

            f1 = (
                2 * precision * recall
                / (precision + recall)
                if (precision + recall) > 0
                else 0
            )

            candidate = {
                "threshold": float(threshold),
                "precision": precision,
                "recall": recall,
                "f1": f1
            }

            if best is None:
                best = candidate

            elif (
                candidate["f1"],
                candidate["recall"],
                -candidate["threshold"]
            ) > (
                best["f1"],
                best["recall"],
                -best["threshold"]
            ):
                best = candidate

        thresholds[category] = best["threshold"]

        results.append({
            "category": category,
            **best
        })

    return thresholds, pd.DataFrame(results)

In [ ]:
thresholds_a, threshold_results_a = select_thresholds(
    scores_a,
    labels_a,
    masks_a,
    category_order
)

thresholds_b, threshold_results_b = select_thresholds(
    scores_b,
    labels_b,
    masks_b,
    category_order
)

In [ ]:
print("Candidate A")
display(threshold_results_a)

print("Candidate B")
display(threshold_results_b)

### Threshold Selection Results

Candidate A achieved a validation macro-F1 of approximately **0.7638**, while Candidate B achieved approximately **0.7675**.

Class weighting improved F1 for several lower-support categories, particularly Change Of Control, Exclusivity, and Non-Compete. However, it also reduced performance for some categories, including Renewal Term and Non-Transferable License. The results therefore show a small overall macro-F1 improvement from weighting, but the effect varies substantially by category.

Final candidate selection will also consider micro-F1 and the full confusion counts rather than macro-F1 alone.

In [ ]:
def evaluate_candidate(
    scores,
    labels,
    masks,
    thresholds,
    category_order
):
    rows = []

    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_tn = 0

    for i, category in enumerate(category_order):

        usable = masks[:, i] == 0

        y_true = labels[usable, i]

        threshold = thresholds[category]

        y_pred = (
            scores[usable, i] >= threshold
        )

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()
        tn = ((y_true == 0) & (y_pred == 0)).sum()

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        rows.append({
            "category": category,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_tn += tn

    results = pd.DataFrame(rows)

    macro_f1 = results["f1"].mean()

    micro_precision = (
        total_tp / (total_tp + total_fp)
    )

    micro_recall = (
        total_tp / (total_tp + total_fn)
    )

    micro_f1 = (
        2 * micro_precision * micro_recall
        / (micro_precision + micro_recall)
    )
    accuracy = (
    (total_tp + total_tn)
    / (total_tp + total_tn + total_fp + total_fn)
    )

    summary = {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "TN": total_tn
    }

    return results, summary

In [ ]:
results_a, summary_a = evaluate_candidate(
    scores_a,
    labels_a,
    masks_a,
    thresholds_a,
    category_order
)

results_b, summary_b = evaluate_candidate(
    scores_b,
    labels_b,
    masks_b,
    thresholds_b,
    category_order
)

print("Candidate A:")
print(summary_a)

print("\nCandidate B:")
print(summary_b)

## 2.17 Candidate Comparison Results

Candidate A and Candidate B produced very similar overall validation performance.

- **Candidate A (unweighted):** macro-F1 = **0.7638**, micro-F1 = **0.8083**
- **Candidate B (weighted):** macro-F1 = **0.7675**, micro-F1 = **0.8056**

Class weighting slightly increased recall from **0.7854 to 0.7906** and reduced false negatives from **288 to 281**, but precision decreased from **0.8325 to 0.8212** and false positives increased from **212 to 231**.

The weighted candidate therefore detected slightly more positive clauses, while the unweighted candidate produced fewer false-positive detections. The macro-F1 difference is small, indicating that neither candidate clearly dominates across all categories.

In [ ]:
comparison = (
    results_a[
        ["category", "precision", "recall", "f1"]
    ]
    .merge(
        results_b[
            ["category", "precision", "recall", "f1"]
        ],
        on="category",
        suffixes=("_a", "_b")
    )
)

comparison["precision_change"] = (
    comparison["precision_b"]
    - comparison["precision_a"]
)

comparison["recall_change"] = (
    comparison["recall_b"]
    - comparison["recall_a"]
)

comparison["f1_change"] = (
    comparison["f1_b"]
    - comparison["f1_a"]
)

comparison["better_candidate"] = np.where(
    comparison["f1_change"] > 0,
    "B - Weighted",
    "A - Unweighted"
)

comparison

## 2.18 Effect of Class Weighting

Class weighting had a category-specific effect rather than producing a uniform improvement.

Candidate B improved F1 most clearly for **Change Of Control (+0.0486)**, **Exclusivity (+0.0312)**, and **Non-Compete (+0.0511)**. Candidate A performed better for **Renewal Term (-0.0473 for B)** and **Non-Transferable License (-0.0255 for B)**. Several other categories changed very little.

Overall, Candidate B achieved slightly higher macro-F1 (**0.7675 vs. 0.7638**) and recall (**0.7906 vs. 0.7854**), while Candidate A achieved higher micro-F1 (**0.8083 vs. 0.8056**) and precision (**0.8325 vs. 0.8212**). Candidate B detected seven additional true positives but also produced nineteen additional false positives.

The imbalance treatment therefore improved detection for some rare categories, but did not clearly dominate the unweighted model across all categories.

## 2.19 Freeze Selected Transformer

Candidate A, the unweighted ModernBERT model, is selected as the final transformer candidate.

Candidate A achieved higher micro-F1 and precision, fewer false positives, and lower validation loss than Candidate B, while Candidate B's macro-F1 improvement was small. Class weighting improved several rare categories but did not consistently improve performance across all 10 categories.

The selected Candidate A checkpoint, tokenizer, category order, preprocessing configuration, and validation-selected category thresholds are frozen for subsequent verification and evaluation.

In [ ]:
import json
from pathlib import Path

FINAL_MODEL_DIR = OUTPUT_DIR / "selected_transformer_candidate_a"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save selected model and tokenizer
best_model_a.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Saved model and tokenizer to:")
print(FINAL_MODEL_DIR)

NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
thresholds_path = FINAL_MODEL_DIR / "thresholds.json"

with open(thresholds_path, "w") as f:
    json.dump(
        thresholds_a,
        f,
        indent=2
    )

print("Saved thresholds:")
print(thresholds_path)

In [ ]:
category_order_path = FINAL_MODEL_DIR / "category_order.json"

with open(category_order_path, "w") as f:
    json.dump(
        category_order,
        f,
        indent=2
    )

print("Saved category order.")

In [ ]:
metadata = {
    "selected_candidate": "A - Unweighted",
    "base_model": MODEL_NAME,

    "best_epoch": 1,
    "best_validation_loss": 0.0801,

    "validation_macro_f1": float(summary_a["macro_f1"]),
    "validation_micro_f1": float(summary_a["micro_f1"]),
    "validation_precision": float(summary_a["micro_precision"]),
    "validation_recall": float(summary_a["micro_recall"]),

    "validation_tp": int(summary_a["TP"]),
    "validation_fp": int(summary_a["FP"]),
    "validation_fn": int(summary_a["FN"]),
    "validation_tn": int(summary_a["TN"]),

    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": NUM_EPOCHS,
    "seed": SEED,

    "chunk_size": 768,
    "chunk_overlap": 512,

    "num_categories": len(category_order),

    "selection_note": (
        "Candidate A was selected because it achieved higher micro-F1 "
        "and precision, fewer false positives, and lower validation loss. "
        "Candidate B achieved slightly higher macro-F1 and recall, but "
        "did not consistently improve performance across categories."
    )
}

metadata_path = FINAL_MODEL_DIR / "metadata.json"

with open(metadata_path, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print("Saved metadata.")

In [ ]:
for file in sorted(FINAL_MODEL_DIR.iterdir()):
    print(file.name)

## 2.20 Frozen Package Verification

The selected Candidate A package is hashed so that the exact model, tokenizer, thresholds, category order, and metadata used for later evaluation can be identified and reproduced.


In [ ]:
import hashlib
import json

def sha256_file(path):
    sha = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(block)

    return sha.hexdigest()


file_hashes = {}

for path in sorted(FINAL_MODEL_DIR.iterdir()):

    if path.is_file():
        file_hashes[path.name] = sha256_file(path)


hashes_path = FINAL_MODEL_DIR / "file_hashes.json"

with open(hashes_path, "w") as f:
    json.dump(
        file_hashes,
        f,
        indent=2
    )


for filename, file_hash in file_hashes.items():
    print(filename, file_hash)

## 2.21 Label-Free Scorer

A label-free scorer is created using the frozen Candidate A package. The scorer receives only the chunk ID and source text and produces one probability score and one binary detection for each of the 10 approved categories.

Gold labels and masks are not used during inference.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import json

frozen_model = (
    AutoModelForSequenceClassification
    .from_pretrained(FINAL_MODEL_DIR)
    .to(device)
)

frozen_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

with open(FINAL_MODEL_DIR / "thresholds.json") as f:
    frozen_thresholds = json.load(f)

with open(FINAL_MODEL_DIR / "category_order.json") as f:
    frozen_category_order = json.load(f)

frozen_model.eval()

print("Frozen package loaded successfully.")

c:\Users\m_laibhen2024\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'FINAL_MODEL_DIR' is not defined

In [ ]:
print("FINAL_MODEL_DIR exists:", "FINAL_MODEL_DIR" in globals())
print("frozen_model exists:", "frozen_model" in globals())
print("transformer_predictions exists:", "transformer_predictions" in globals())

In [ ]:
def score_chunks(
    chunks_df,
    model,
    tokenizer,
    thresholds,
    category_order,
    device
):
    rows = []

    model.eval()

    for _, row in tqdm(
        chunks_df.iterrows(),
        total=len(chunks_df),
        desc="Scoring"
    ):

        encoded = tokenizer(
            row["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            scores = torch.sigmoid(
                outputs.logits
            )[0].cpu().numpy()

        for i, category in enumerate(category_order):

            threshold = thresholds[category]

            rows.append({
                "chunk_id": row["chunk_id"],
                "category": category,
                "score": float(scores[i]),
                "threshold": float(threshold),
                "detection": int(
                    scores[i] >= threshold
                )
            })

    return pd.DataFrame(rows)

In [ ]:
test_chunks = val_examples[
    ["chunk_id", "text"]
].head(2)

test_predictions = score_chunks(
    test_chunks,
    frozen_model,
    frozen_tokenizer,
    frozen_thresholds,
    frozen_category_order,
    device
)

test_predictions

NameError: name 'val_examples' is not defined

# Gate 3.1 — Evaluation Alignment Report

## Objective

Verify that the October transformer and September keyword and TF-IDF baselines use identical approved validation evaluation units.

## Evaluation Configuration

- Validation chunks: 2,481
- Categories: 10
- Total evaluation pairs: 24,810
- Unmasked pairs: 24,646
- Masked pairs: 164
- Chunk size: 768 tokens
- Overlap: 512 tokens

## September Baseline Verification

**Status: PASS**

- Keyword: All 24,810 validation scores reproduced exactly.
- TF-IDF: All 24,646 unmasked margins reproduced within numerical tolerance, with identical detections.
- Both baselines use identical `(chunk_id, category)` pairs, targets, masks, document titles, and context group IDs.

## October Transformer Verification

**Status: PASS**

- The frozen Candidate A transformer generated 24,810 predictions across 2,481 validation chunks.
- All predicted `(chunk_id, category)` pairs match the approved validation labels.
- Every prediction maps to an approved label, mask, and source record.
- The transformer uses the same 10 category names and approved validation text.
- Category comparisons use names rather than positional order.

## Conclusion

The completed checks support evaluation-unit alignment between the October transformer and both September baselines.

No evaluation-unit differences were identified by these checks. September baseline results were preserved without retraining or retuning.

## Remaining Work

- Record the repository commit SHA.
- Record the frozen transformer model artifact location.
- Generate SHA-256 hashes for evaluation artifacts.
- Obtain independent review if required.

**Overall status:** Alignment checks passed; final reproducibility documentation pending.

In [ ]:
# Step 6A: Score all validation chunks using frozen Candidate A

full_validation_chunks = val_examples[
    ["chunk_id", "text"]
].copy()

transformer_predictions = score_chunks(
    full_validation_chunks,
    frozen_model,
    frozen_tokenizer,
    frozen_thresholds,
    frozen_category_order,
    device
)

print("Total validation chunks:", len(full_validation_chunks))
print("Total transformer predictions:", len(transformer_predictions))

assert len(full_validation_chunks) == 2481
assert len(transformer_predictions) == 24810

print("PASS: Transformer scored all validation chunks.")

NameError: name 'val_examples' is not defined

In [ ]:
# Step 6D: Verify category order and input text

# 1. Check that category names match
approved_categories = set(val_labels["category"].unique())
transformer_categories = set(frozen_category_order)

assert len(frozen_category_order) == len(approved_categories)
assert len(frozen_category_order) == len(set(frozen_category_order))
assert transformer_categories == approved_categories

print("PASS: Transformer category names match approved categories.")

# 2. Check that prediction categories match the frozen model order
assert set(transformer_predictions["category"]) == transformer_categories

# 3. Verify the text supplied to the scorer matches approved chunks
text_comparison = full_validation_chunks.merge(
    val_chunks[["chunk_id", "text"]],
    on="chunk_id",
    suffixes=("_scored", "_approved"),
    validate="one_to_one"
)

assert len(text_comparison) == len(val_chunks)
assert (
    text_comparison["text_scored"]
    == text_comparison["text_approved"]
).all()

print("PASS: Transformer scored the approved validation text.")
print("Categories:", len(frozen_category_order))
print("Validated chunks:", len(text_comparison))

In [ ]:
# Step 6B: Compare transformer predictions to approved validation labels

keys = ["chunk_id", "category"]

# Ensure there are no duplicate predictions
assert not transformer_predictions.duplicated(keys).any()

# Ensure the approved labels have no duplicate pairs
assert not val_labels.duplicated(keys).any()

transformer_keys = set(
    map(tuple, transformer_predictions[keys].to_numpy())
)

approved_keys = set(
    map(tuple, val_labels[keys].to_numpy())
)

missing = approved_keys - transformer_keys
extra = transformer_keys - approved_keys

print("Missing evaluation pairs:", len(missing))
print("Extra evaluation pairs:", len(extra))

assert not missing, "Transformer is missing approved evaluation pairs"
assert not extra, "Transformer contains unexpected evaluation pairs"

print("PASS: Transformer uses the exact approved chunk-category pairs.")

In [ ]:
from pathlib import Path

output_dir = Path("reports/gate3_alignment")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "transformer_validation_predictions.csv"

transformer_predictions.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(transformer_predictions))
print("Columns:", transformer_predictions.columns.tolist())

NameError: name 'transformer_predictions' is not defined

In [105]:
variables = [
    "frozen_model",
    "frozen_tokenizer",
    "frozen_thresholds",
    "frozen_category_order",
    "val_examples",
    "score_chunks",
    "device",
]

for name in variables:
    print(f"{name}: {name in globals()}")

frozen_model: False
frozen_tokenizer: False
frozen_thresholds: False
frozen_category_order: False
val_examples: True
score_chunks: False
device: True


In [106]:
from pathlib import Path

print("Current directory:", Path.cwd())

model_dirs = list(
    Path.cwd().rglob("selected_transformer_candidate_a")
)

for path in model_dirs:
    print("Found:", path)
    print("Files:", [p.name for p in path.iterdir()])

Current directory: c:\Users\m_laibhen2024\Accenture-1D-contract-review-challenge\notebooks


In [107]:
from pathlib import Path

repo_root = Path.cwd().parent

print("Repository root:", repo_root)

model_dirs = list(
    repo_root.rglob("selected_transformer_candidate_a")
)

if not model_dirs:
    print("No Candidate A model folder found.")
else:
    for folder in model_dirs:
        print("\nFound:", folder)
        print("Files:")
        for file in folder.iterdir():
            print(" -", file.name)

Repository root: c:\Users\m_laibhen2024\Accenture-1D-contract-review-challenge

Found: c:\Users\m_laibhen2024\Accenture-1D-contract-review-challenge\configs\Accenture_October_Transformer\selected_transformer_candidate_a
Files:
 - category_order.json
 - config.json
 - file_hashes.json
 - metadata.json
 - thresholds.json
 - tokenizer.json
 - tokenizer_config.json


In [ ]:
print("Total rows:", len(test_predictions))

print(
    "Outputs per chunk:",
    test_predictions.groupby(
        "chunk_id"
    ).size().unique()
)

print(
    "All scores finite:",
    np.isfinite(
        test_predictions["score"]
    ).all()
)

print(
    "Categories:",
    test_predictions["category"].unique()
)

NameError: name 'test_predictions' is not defined

In [ ]:
print("Predictions available:", "transformer_predictions" in globals())

if "transformer_predictions" in globals():
    print("Prediction rows:", len(transformer_predictions))

Predictions available: False


## 2.22 Trainable Parameters

The ModernBERT encoder and classification head were fine-tuned together. The total and trainable parameter counts are recorded for reproducibility.

In [ ]:
total_params = sum(
    p.numel()
    for p in frozen_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in frozen_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Percent trainable: "
    f"{100 * trainable_params / total_params:.2f}%"
)

In [ ]:
print("Train chunk columns:")
print(train_chunks.columns.tolist())

print("\nValidation chunk columns:")
print(val_chunks.columns.tolist())

print("\nTrain label columns:")
print(train_labels.columns.tolist())

In [ ]:
print("Using split manifest from the repository data directory.")

In [ ]:
import pandas as pd
split_manifest = pd.read_csv(
    NOTEBOOK_DIR / "data" / "splits" / "split_manifest.csv"
)

print(split_manifest.columns.tolist())
split_manifest.head()

In [ ]:
train_labels_grouped = train_labels.merge(
    split_manifest[
        ["title", "context_group_id"]
    ].drop_duplicates(),
    on="title",
    how="left"
)

val_labels_grouped = val_labels.merge(
    split_manifest[
        ["title", "context_group_id"]
    ].drop_duplicates(),
    on="title",
    how="left"
)

NameError: name 'train_labels' is not defined

In [ ]:
print(
    "Missing train groups:",
    train_labels_grouped["context_group_id"].isna().sum()
)

print(
    "Missing validation groups:",
    val_labels_grouped["context_group_id"].isna().sum()
)

In [ ]:
train_group_support = (
    train_labels_grouped[
        (train_labels_grouped["mask"] == 0)
        & (train_labels_grouped["target"] == 1)
    ]
    .groupby("category")["context_group_id"]
    .nunique()
    .reset_index(name="train_positive_context_groups")
)

val_group_support = (
    val_labels_grouped[
        (val_labels_grouped["mask"] == 0)
        & (val_labels_grouped["target"] == 1)
    ]
    .groupby("category")["context_group_id"]
    .nunique()
    .reset_index(name="validation_positive_context_groups")
)

group_support = train_group_support.merge(
    val_group_support,
    on="category"
)

group_support